# Proyecto BI — Ventas de Retail (Superstore Sales)
**Institución Universitaria ITM · Análisis de Datos · 2026-1**

**Dataset:** Superstore Sales (`train.csv`) — 9 800 registros de órdenes de venta
al por menor en Estados Unidos (2015-2018), con variables de cliente, producto,
envío y ventas.

In [ ]:
# ─── Importaciones globales ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (10, 5)

# Carga del dataset
df = pd.read_csv("train.csv", sep=None, engine="python", encoding="latin1")

print(f"Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}")
print(df.dtypes)

---
## Fase 1 · Comprensión del Problema

### 1.1 Descripción del contexto del dataset

El dataset **Superstore Sales** contiene registros de órdenes realizadas en una
cadena minorista estadounidense entre 2015 y 2018. Cada fila representa una línea
de pedido e incluye información sobre:

| Dimensión | Variables |
|-----------|-----------|
| Tiempo | `Order Date`, `Ship Date` |
| Cliente | `Customer ID`, `Customer Name`, `Segment` |
| Geografía | `Country`, `City`, `State`, `Postal Code`, `Region` |
| Logística | `Ship Mode` |
| Producto | `Product ID`, `Category`, `Sub-Category`, `Product Name` |
| Finanzas | `Sales` |

### 1.2 Identificación del problema de negocio

La empresa necesita entender **qué factores impulsan o frenan las ventas** para
optimizar la asignación de recursos comerciales, logísticos y de marketing.
En concreto, se desconoce si existen diferencias significativas entre regiones,
segmentos de clientes y modalidades de envío que justifiquen estrategias
diferenciadas.

### 1.3 Preguntas clave

1. **¿Qué región, categoría y segmento de clientes generan más ventas?**
2. **¿El modo de envío influye en el valor promedio de la venta?**
3. **¿Existen diferencias estadísticamente significativas entre las ventas
   medianas de los distintos segmentos de clientes (Consumer, Corporate, Home Office)?**
4. **¿Qué productos o sub-categorías concentran el mayor volumen de ingresos?**
5. **¿Cómo evolucionan las ventas a lo largo del tiempo?**

In [ ]:
# ─── Fase 1 · Exploración inicial ─────────────────────────────────────────────

print("=" * 60)
print("FASE 1 — Exploración inicial del dataset")
print("=" * 60)

print("\n── Primeras filas ──")
print(df.head(3).to_string())

print("\n── Estadísticas descriptivas de Sales ──")
print(df["Sales"].describe().round(2))
print(f"\n  Skewness: {df['Sales'].skew():.2f}  (>1 = sesgo alto, distribución no normal)")
print(f"  Media:    ${df['Sales'].mean():.2f}")
print(f"  Mediana:  ${df['Sales'].median():.2f}  → ratio media/mediana = {df['Sales'].mean()/df['Sales'].median():.1f}x")

print("\n── Distribución por Segmento ──")
print(df["Segment"].value_counts())

print("\n── Distribución por Región ──")
print(df["Region"].value_counts())

print("\n── Distribución por Categoría ──")
print(df["Category"].value_counts())

print("\n── Distribución por Ship Mode ──")
print(df["Ship Mode"].value_counts())

### Decisión metodológica — Transformación log(Sales + 1)

Al explorar la variable `Sales` encontramos los siguientes valores:

| Estadístico | Valor |
|-------------|-------|
| Media | $230.77 |
| Mediana | $54.49 |
| Ratio media/mediana | 4.2× |
| Skewness | **12.98** |
| Máximo | $22,638.48 |

Un **skewness de 12.98** es extremadamente alto (valores >1 ya se consideran
sesgados; >3 son severos). Esto ocurre porque el 4.7 % de los pedidos
(aquellos con `Sales > $1,000`) concentran el **43.2 % de los ingresos totales**:
pocas ventas de gran valor "estiran" la cola derecha de la distribución.

**¿Por qué log(Sales + 1) y no otra transformación?**

- **`log(x + 1)`** (logaritmo natural con corrección de +1 para evitar log(0)):
  comprime la cola larga de forma proporcional, convirtiendo diferencias
  multiplicativas en diferencias aditivas. Es la transformación estándar para
  variables monetarias con sesgo positivo.
- Alternativas como raíz cuadrada o Box-Cox también reducen el sesgo, pero
  `log1p` es más interpretable: un incremento de 1 unidad en la escala log
  equivale aproximadamente a un incremento porcentual constante en Sales.
- **Resultado:** el skewness pasa de 12.98 → **0.28**, obteniendo una
  distribución prácticamente normal (campana simétrica visible en el histograma).

> **Nota importante:** la transformación se aplica **solo para visualización**.
> Los KPIs y agregados de negocio (sumas, medianas) se calculan siempre sobre
> los valores originales de `Sales`, que son los dólares reales.

In [ ]:
# ─── Fase 1 · Visualización exploratoria ──────────────────────────────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Fase 1 — Distribución general del dataset", fontsize=14, fontweight="bold")

# Ventas por Segmento
seg_sales = df.groupby("Segment")["Sales"].sum().sort_values(ascending=False)
axes[0, 0].bar(seg_sales.index, seg_sales.values, color=sns.color_palette("Set2", 3))
axes[0, 0].set_title("Ventas totales por Segmento")
axes[0, 0].set_ylabel("Ventas (USD)")
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Ventas por Región
reg_sales = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
axes[0, 1].bar(reg_sales.index, reg_sales.values, color=sns.color_palette("Set2", 4))
axes[0, 1].set_title("Ventas totales por Región")
axes[0, 1].set_ylabel("Ventas (USD)")
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Ventas por Categoría
cat_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
axes[1, 0].bar(cat_sales.index, cat_sales.values, color=sns.color_palette("Set2", 3))
axes[1, 0].set_title("Ventas totales por Categoría")
axes[1, 0].set_ylabel("Ventas (USD)")
axes[1, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

log_sales = np.log1p(df["Sales"])
axes[1, 1].hist(log_sales, bins=50, color="#4C72B0", edgecolor="white")
axes[1, 1].set_title(
    f"Distribución de log(Sales+1)\n"
    f"Skewness original={df['Sales'].skew():.1f} → log={log_sales.skew():.2f}"
)
axes[1, 1].set_xlabel("log(Ventas + 1)")
axes[1, 1].set_ylabel("Frecuencia")
# Ticks en escala original para referencia
tick_vals = [0, 10, 50, 100, 500, 1000, 5000, 22000]
axes[1, 1].set_xticks([np.log1p(v) for v in tick_vals])
axes[1, 1].set_xticklabels([f"${v:,.0f}" for v in tick_vals], rotation=35, ha="right", fontsize=7)

plt.tight_layout()
plt.savefig("fase1_exploracion.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fase1_exploracion.png")

---
## Fase 2 · Formulación de Hipótesis

### Decisión metodológica — ¿Por qué Kruskal-Wallis y no ANOVA?

La elección del test estadístico depende de si los datos cumplen ciertos
supuestos. Para comparar grupos con una variable continua existen dos caminos:

| Criterio | ANOVA (paramétrico) | Kruskal-Wallis (no paramétrico) |
|----------|--------------------|---------------------------------|
| Supuesto principal | Distribución **normal** dentro de cada grupo | **Ninguno** sobre la distribución |
| Qué compara | Medias | Distribuciones (rangos) |
| Sensible a outliers | **Sí** — la media se distorsiona | No — trabaja con rangos |
| Válido para Sales | ❌ — skewness = 12.98 | ✅ |

**¿Por qué Sales viola el supuesto de ANOVA?**

ANOVA asume que la variable de respuesta sigue una distribución aproximadamente
normal dentro de cada grupo. `Sales` tiene skewness = **12.98**: los valores
extremos (pedidos de $5,000–$22,000) inflan enormemente la varianza dentro de
cada grupo, haciendo que el estadístico F de ANOVA pierda potencia y produzca
p-values poco confiables.

**¿Qué hace Kruskal-Wallis?**

En lugar de comparar las medias directamente, convierte todos los valores de
`Sales` en **rangos** (1 = el pedido más barato, 9800 = el más caro) y evalúa
si los rangos se distribuyen de forma similar entre los grupos. Al trabajar
con rangos, los valores extremos dejan de distorsionar el análisis: un pedido
de $22,000 simplemente tiene el rango más alto, pero no "pesa" más que eso.

**¿Cambia la conclusión?**

En este dataset, **no**: tanto ANOVA como Kruskal-Wallis concluyen que no hay
diferencias significativas entre segmentos (p=0.80) ni entre modos de envío
(p=0.58). Sin embargo, usar Kruskal-Wallis es la decisión **metodológicamente
correcta y académicamente defendible**, independientemente del resultado.
Si los grupos sí tuvieran diferencias reales, ANOVA con datos tan sesgados
podría haberlas ocultado (error de tipo II).

### Hipótesis 1 — Diferencia en ventas medianas entre segmentos

| | Enunciado |
|---|---|
| **H₀** | La distribución de ventas es igual en los tres segmentos (Consumer, Corporate, Home Office). |
| **H₁** | Al menos un segmento tiene una distribución de ventas significativamente diferente. |

**Justificación:** Si los segmentos presentan comportamientos de compra distintos,
la empresa puede diseñar estrategias de precios y promociones diferenciadas para
maximizar el ingreso por cliente.

**Test:** **Kruskal-Wallis** (no paramétrico). Se descarta ANOVA porque la variable
`Sales` tiene skewness = 12.98, violando el supuesto de normalidad. Kruskal-Wallis
es robusto ante distribuciones sesgadas y compara distribuciones entre grupos.

---
### Hipótesis 2 — Diferencia en ventas medianas entre modos de envío

| | Enunciado |
|---|---|
| **H₀** | La distribución de ventas es igual para los cuatro modos de envío. |
| **H₁** | El modo de envío "Same Day" está asociado a una distribución de ventas significativamente diferente. |

**Justificación:** Los envíos urgentes suelen implicar artículos de menor tamaño
o valor. Confirmar esto permite al equipo logístico priorizar capacidad en los
modos más rentables.

**Test:** **Kruskal-Wallis** (misma justificación que H1).

In [ ]:
# ─── Fase 2 · Preparación de datos para los tests ─────────────────────────────

print("=" * 60)
print("FASE 2 — Formulación y validación de hipótesis")
print("=" * 60)

alpha = 0.05

# Grupos por segmento
consumer    = df[df["Segment"] == "Consumer"]["Sales"]
corporate   = df[df["Segment"] == "Corporate"]["Sales"]
home_office = df[df["Segment"] == "Home Office"]["Sales"]

# Grupos por Ship Mode
first_class  = df[df["Ship Mode"] == "First Class"]["Sales"]
second_class = df[df["Ship Mode"] == "Second Class"]["Sales"]
standard     = df[df["Ship Mode"] == "Standard Class"]["Sales"]
same_day     = df[df["Ship Mode"] == "Same Day"]["Sales"]

In [ ]:
# ─── Fase 2 · Hipótesis 1 — Kruskal-Wallis por Segmento ──────────────────────

print("\n── HIPÓTESIS 1: Diferencia de ventas entre segmentos ──")
print(f"  {'Segmento':<15} {'Mediana':>10}  {'Media':>10}  {'n':>6}")
print(f"  {'-'*45}")
for name, grp in [("Consumer", consumer), ("Corporate", corporate), ("Home Office", home_office)]:
    print(f"  {name:<15} ${grp.median():>9,.2f}  ${grp.mean():>9,.2f}  {len(grp):>6}")

h_stat_seg, p_val_seg = stats.kruskal(consumer, corporate, home_office)
print(f"\n  Test: Kruskal-Wallis")
print(f"  H-statistic = {h_stat_seg:.4f}")
print(f"  p-value     = {p_val_seg:.6f}")

if p_val_seg < alpha:
    print(f"\n  ✔ Se RECHAZA H₀ (p < {alpha}): existen diferencias significativas")
    print("    en la distribución de ventas entre segmentos.")
else:
    print(f"\n  ✘ No se rechaza H₀ (p ≥ {alpha}): no hay diferencias significativas")
    print("    en la distribución de ventas entre segmentos.")

In [ ]:
# ─── Fase 2 · Hipótesis 2 — Kruskal-Wallis por Ship Mode ─────────────────────

print("\n── HIPÓTESIS 2: Diferencia de ventas entre modos de envío ──")
print(f"  {'Ship Mode':<20} {'Mediana':>10}  {'Media':>10}  {'n':>6}")
print(f"  {'-'*50}")
for mode in ["First Class", "Second Class", "Standard Class", "Same Day"]:
    grp = df[df["Ship Mode"] == mode]["Sales"]
    print(f"  {mode:<20} ${grp.median():>9,.2f}  ${grp.mean():>9,.2f}  {len(grp):>6}")

h_stat_ship, p_val_ship = stats.kruskal(first_class, second_class, standard, same_day)
print(f"\n  Test: Kruskal-Wallis")
print(f"  H-statistic = {h_stat_ship:.4f}")
print(f"  p-value     = {p_val_ship:.6f}")

if p_val_ship < alpha:
    print(f"\n  ✔ Se RECHAZA H₀ (p < {alpha}): el modo de envío afecta")
    print("    significativamente la distribución de ventas.")
else:
    print(f"\n  ✘ No se rechaza H₀ (p ≥ {alpha}): el modo de envío no afecta")
    print("    significativamente la distribución de ventas.")

In [ ]:
# ─── Fase 2 · Visualización de hipótesis ──────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Fase 2 — Distribución de ventas por grupo (escala log)", fontsize=14, fontweight="bold")

# Box-plot segmentos — escala log
sns.boxplot(
    data=df, x="Segment", y="Sales", ax=axes[0],
    palette="Set2", order=["Consumer", "Corporate", "Home Office"],
    flierprops=dict(marker=".", markersize=2, alpha=0.3)
)
axes[0].set_yscale("log")
axes[0].set_title(f"Ventas por Segmento\n(Kruskal-Wallis  H={h_stat_seg:.3f},  p={p_val_seg:.4f})")
axes[0].set_xlabel("Segmento")
axes[0].set_ylabel("Ventas (USD) — escala log")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for i, grp in enumerate([consumer, corporate, home_office]):
    med = grp.median()
    axes[0].text(i, med * 1.15, f"${med:.0f}", ha="center", va="bottom",
                 fontsize=8, fontweight="bold", color="navy")

# Box-plot ship mode — escala log
order_ship = ["Same Day", "First Class", "Second Class", "Standard Class"]
sns.boxplot(
    data=df, x="Ship Mode", y="Sales", ax=axes[1],
    palette="Set2", order=order_ship,
    flierprops=dict(marker=".", markersize=2, alpha=0.3)
)
axes[1].set_yscale("log")
axes[1].set_title(f"Ventas por Ship Mode\n(Kruskal-Wallis  H={h_stat_ship:.3f},  p={p_val_ship:.4f})")
axes[1].set_xlabel("Ship Mode")
axes[1].set_ylabel("Ventas (USD) — escala log")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for i, mode in enumerate(order_ship):
    med = df[df["Ship Mode"] == mode]["Sales"].median()
    axes[1].text(i, med * 1.15, f"${med:.0f}", ha="center", va="bottom",
                 fontsize=8, fontweight="bold", color="darkgreen")

plt.tight_layout()
plt.savefig("fase2_hipotesis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fase2_hipotesis.png")

---
## Fase 3 · Preparación y Modelado de Datos

En esta fase se realiza la **limpieza**, **transformación** y **estructuración**
del dataset para dejarlo listo para el análisis y el dashboard.

In [ ]:
# ─── Fase 3 · Limpieza: nulos y duplicados ────────────────────────────────────

print("=" * 60)
print("FASE 3 — Preparación y modelado de datos")
print("=" * 60)

print("\n── Valores nulos por columna ──")
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else "  Sin valores nulos")
print(f"  (Postal Code: 11 nulos — no afecta el análisis de ventas)")

print(f"\n── Filas duplicadas: {df.duplicated().sum()} ──")
df = df.drop_duplicates()
print(f"Filas tras eliminar duplicados: {len(df)}")

In [ ]:
# ─── Fase 3 · Transformación de tipos y variables derivadas ───────────────────

# Convertir fechas
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True)
df["Ship Date"]  = pd.to_datetime(df["Ship Date"],  dayfirst=True)

# Lead Time (días entre orden y envío)
df["Lead Time Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

# Componentes temporales
df["Order Year"]    = df["Order Date"].dt.year
df["Order Month"]   = df["Order Date"].dt.month
df["Order Quarter"] = df["Order Date"].dt.to_period("Q").astype(str)

# Transformación logarítmica de Sales (para análisis y visualizaciones)
df["log_Sales"] = np.log1p(df["Sales"])

print("\n── Nuevas columnas derivadas ──")
print(df[["Order Date", "Ship Date", "Lead Time Days",
          "Order Year", "Order Month", "Order Quarter", "log_Sales"]].head(5))

print(f"\n── Lead Time: verificación ──")
print(df.groupby("Ship Mode")["Lead Time Days"].describe().round(2))
print(f"\n  Lead Time = 0: {(df['Lead Time Days'] == 0).sum()} registros "
      f"→ todos son 'Same Day' (correcto, no son errores)")

In [ ]:
# ─── Fase 3 · Selección de variables ──────────────────────────────────────────

DIMENSIONS = [
    "Order ID", "Order Date", "Ship Date",
    "Ship Mode", "Customer ID", "Segment",
    "Region", "State", "City",
    "Category", "Sub-Category",
    "Order Year", "Order Month", "Order Quarter",
    "Lead Time Days"
]
METRICS = ["Sales", "log_Sales"]

print("\n── Dimensiones seleccionadas ──")
print(DIMENSIONS)
print("\n── Métricas seleccionadas ──")
print(METRICS)

In [ ]:
# ─── Fase 3 · Estructuración: tablas para análisis ────────────────────────────

# Tabla agregada por Order ID
order_summary = (
    df.groupby(["Order ID", "Order Date", "Ship Mode", "Segment", "Region",
                "State", "Category", "Order Year", "Order Quarter"])
    .agg(
        Total_Sales=("Sales", "sum"),
        Num_Items=("Sales", "count"),
        Avg_Item_Sales=("Sales", "mean"),
        Lead_Time=("Lead Time Days", "first")
    )
    .reset_index()
)

print(f"\n── Tabla de órdenes agregadas: {len(order_summary)} registros ──")
print(order_summary.head(3).to_string())

# Tabla cruzada Segmento × Categoría
seg_cat_summary = (
    df.groupby(["Segment", "Category"])["Sales"]
    .agg(Total_Sales="sum", Median_Sales="median", Avg_Sales="mean", Count="count")
    .reset_index()
)
print(f"\n── Ventas por Segmento × Categoría ──")
print(seg_cat_summary.to_string())

In [ ]:
# ─── Fase 3 · Outliers en Sales ───────────────────────────────────────────────

Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = df[df["Sales"] > upper_fence]
print(f"\n── Outliers en Sales (IQR) ──")
print(f"  Límite superior (Q3 + 1.5·IQR): ${upper_fence:,.2f}")
print(f"  Outliers: {len(outliers)} registros ({len(outliers)/len(df)*100:.1f}%)")
print(f"  Su aporte al ingreso total: {outliers['Sales'].sum()/df['Sales'].sum()*100:.1f}%")
print("  → Se CONSERVAN: son ventas B2B legítimas de alto valor.")
print("    Para visualizaciones se usa escala log para no distorsionar.")

In [ ]:
# ─── Fase 3 · Visualización de datos transformados ────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Fase 3 — Datos transformados y estructurados", fontsize=14, fontweight="bold")

# Evolución mensual de ventas
monthly = df.groupby(["Order Year", "Order Month"])["Sales"].sum().reset_index()
monthly["Periodo"] = pd.to_datetime(
    monthly.rename(columns={"Order Year": "year", "Order Month": "month"})
    [["year", "month"]].assign(day=1)
)
monthly = monthly.sort_values("Periodo")
axes[0].plot(monthly["Periodo"], monthly["Sales"], marker="o", markersize=3, linewidth=1.5)
# Añadir línea de tendencia
z = np.polyfit(range(len(monthly)), monthly["Sales"], 1)
p = np.poly1d(z)
axes[0].plot(monthly["Periodo"], p(range(len(monthly))),
             "--", color="red", linewidth=1, alpha=0.7, label="Tendencia")
axes[0].legend(fontsize=8)
axes[0].set_title("Ventas mensuales (con tendencia)")
axes[0].set_xlabel("Mes")
axes[0].set_ylabel("Ventas (USD)")
axes[0].tick_params(axis="x", rotation=45)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))

# Lead Time por Ship Mode
lt_mode = df.groupby("Ship Mode")["Lead Time Days"].mean().sort_values()
bars = axes[1].barh(lt_mode.index, lt_mode.values, color=sns.color_palette("Set2", len(lt_mode)))
for bar, val in zip(bars, lt_mode.values):
    axes[1].text(val + 0.05, bar.get_y() + bar.get_height() / 2,
                 f"{val:.1f}d", va="center", fontsize=9)
axes[1].set_title("Lead Time promedio por Ship Mode")
axes[1].set_xlabel("Días promedio")
axes[1].set_xlim(0, lt_mode.max() + 1)

# Top 10 Sub-Categorías
subcat = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False).head(10)
axes[2].barh(subcat.index[::-1], subcat.values[::-1],
             color=sns.color_palette("Set2", len(subcat)))
axes[2].set_title("Top 10 Sub-Categorías por ventas")
axes[2].set_xlabel("Ventas (USD)")
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))

plt.tight_layout()
plt.savefig("fase3_modelado.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fase3_modelado.png")

In [ ]:
df.to_csv("dataset_limpio.csv", index=False)

print("\nArchivos guardados:")
print("- dataset_limpio.csv")

---
## Fase 4 · Definición de KPIs

Los KPIs se calculan sobre la tabla limpia y transformada.

| # | KPI | Relación con hipótesis |
|---|-----|------------------------|
| 1 | Ingresos Totales | Base de medición |
| 2 | Ticket Promedio por Orden (por Segmento) | H1 |
| 3 | Venta Mediana por Modo de Envío | H2 |
| 4 | Concentración Pareto 80/20 | Eficiencia del portafolio |
| 5 | Lead Time Promedio por Ship Mode | H2 — eficiencia logística |
| 6 | Participación de Ventas por Región | Priorización geográfica |
| 7 | Tasa de Crecimiento YoY | Tendencia temporal |

In [ ]:
# ─── Fase 4 · Cálculo de KPIs ─────────────────────────────────────────────────

print("=" * 60)
print("FASE 4 — Definición y cálculo de KPIs")
print("=" * 60)

# KPI 1 — Ingresos Totales
kpi1_total_sales = df["Sales"].sum()
print(f"\n[KPI 1] Ingresos Totales: ${kpi1_total_sales:,.2f}")
print("  Métrica base. Sirve de denominador para los KPIs de participación.")

# KPI 2 — Ticket Promedio por Orden (por Segmento)
kpi2_ticket = order_summary.groupby("Segment")["Total_Sales"].mean()
print(f"\n[KPI 2] Ticket Promedio por Orden (por Segmento)")
for seg, val in kpi2_ticket.items():
    print(f"  {seg:<15}: ${val:,.2f}")
print("  H1: ticket mayor en Home Office/Corporate puede justificar fuerza de ventas B2B.")

# KPI 3 — Venta Mediana por Modo de Envío (mediana por robustez ante outliers)
kpi3_ship = df.groupby("Ship Mode")["Sales"].median().sort_values(ascending=False)
print(f"\n[KPI 3] Venta MEDIANA por Modo de Envío")
for mode, val in kpi3_ship.items():
    print(f"  {mode:<20}: ${val:,.2f}")
print("  H2: se usa mediana (robusta al sesgo) en lugar de media.")

# KPI 4 — Pareto
subcat_sales = df.groupby("Sub-Category")["Sales"].sum().sort_values(ascending=False)
total = subcat_sales.sum()
cumulative = subcat_sales.cumsum() / total
n_80 = (cumulative <= 0.80).sum() + 1
print(f"\n[KPI 4] Concentración Pareto: {n_80} de {len(subcat_sales)} sub-categorías = 80% ventas")
print(f"  Top sub-categorías: {', '.join(subcat_sales.head(n_80).index.tolist())}")

# KPI 5 — Lead Time promedio
kpi5_lt = df.groupby("Ship Mode")["Lead Time Days"].mean().sort_values()
print(f"\n[KPI 5] Lead Time Promedio (días)")
for mode, val in kpi5_lt.items():
    print(f"  {mode:<20}: {val:.2f} días")

# KPI 6 — Participación por Región
kpi6_region = df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
kpi6_share  = (kpi6_region / kpi1_total_sales * 100).round(2)
print(f"\n[KPI 6] Participación de Ventas por Región")
for region, share in kpi6_share.items():
    print(f"  {region:<10}: {share:.2f}%  (${kpi6_region[region]:,.2f})")

# KPI 7 — Crecimiento YoY
yearly = df.groupby("Order Year")["Sales"].sum().sort_index()
yoy    = yearly.pct_change() * 100
print(f"\n[KPI 7] Tasa de Crecimiento YoY")
for yr, val in yearly.items():
    g = f"{yoy[yr]:+.1f}%" if not pd.isna(yoy[yr]) else "—"
    print(f"  {yr}: ${val:,.2f}  ({g})")

In [ ]:
# ─── Fase 4 · Visualización de KPIs ───────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Fase 4 — Dashboard de KPIs", fontsize=15, fontweight="bold")

# KPI 1 + KPI 7 — Ventas anuales
colors_yr = sns.color_palette("Set2", len(yearly))
bars = axes[0, 0].bar(yearly.index.astype(str), yearly.values, color=colors_yr)
axes[0, 0].set_title(f"KPI 1 & 7 — Ingresos anuales\nTotal: ${kpi1_total_sales:,.0f}")
axes[0, 0].set_ylabel("USD")
axes[0, 0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}K"))
for bar, yoy_val in zip(bars, yoy.values):
    if not pd.isna(yoy_val):
        axes[0, 0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
                        f"{yoy_val:+.1f}%", ha="center", va="bottom", fontsize=9,
                        color="green" if yoy_val >= 0 else "red", fontweight="bold")

# KPI 2 — Ticket por Segmento
axes[0, 1].bar(kpi2_ticket.index, kpi2_ticket.values,
               color=sns.color_palette("Set2", len(kpi2_ticket)))
axes[0, 1].set_title("KPI 2 — Ticket promedio por Segmento")
axes[0, 1].set_ylabel("USD por orden")
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
for i, (seg, val) in enumerate(kpi2_ticket.items()):
    axes[0, 1].text(i, val + 3, f"${val:,.0f}", ha="center", va="bottom", fontsize=9)

# KPI 3 — Venta MEDIANA por Ship Mode
axes[0, 2].bar(kpi3_ship.index, kpi3_ship.values,
               color=sns.color_palette("Set2", len(kpi3_ship)))
axes[0, 2].set_title("KPI 3 — Venta MEDIANA por Ship Mode")
axes[0, 2].set_ylabel("USD (mediana)")
axes[0, 2].set_xticklabels(kpi3_ship.index, rotation=15, ha="right")
for i, (mode, val) in enumerate(kpi3_ship.items()):
    axes[0, 2].text(i, val + 0.5, f"${val:.1f}", ha="center", va="bottom", fontsize=9)

# KPI 4 — Curva de Pareto
cum_pct = subcat_sales.cumsum() / total * 100
axes[1, 0].bar(range(len(subcat_sales)), subcat_sales.values, color="#AED6F1")
ax4b = axes[1, 0].twinx()
ax4b.plot(range(len(subcat_sales)), cum_pct.values, color="red", marker=".", label="% acumulado")
ax4b.axhline(80, color="orange", linestyle="--", linewidth=1.5, label="80%")
ax4b.axvline(n_80 - 1, color="purple", linestyle=":", linewidth=1.5, label=f"n={n_80}")
ax4b.set_ylabel("% acumulado")
ax4b.set_ylim(0, 105)
ax4b.legend(fontsize=7, loc="lower right")
axes[1, 0].set_title(f"KPI 4 — Pareto ({n_80}/{len(subcat_sales)} sub-cats = 80% ventas)")
axes[1, 0].set_xticks([])
axes[1, 0].set_ylabel("Ventas (USD)")

# KPI 5 — Lead Time
bars5 = axes[1, 1].barh(kpi5_lt.index, kpi5_lt.values,
                         color=sns.color_palette("Set2", len(kpi5_lt)))
for bar, val in zip(bars5, kpi5_lt.values):
    axes[1, 1].text(val + 0.05, bar.get_y() + bar.get_height() / 2,
                    f"{val:.1f}d", va="center", fontsize=9)
axes[1, 1].set_title("KPI 5 — Lead Time promedio por Ship Mode")
axes[1, 1].set_xlabel("Días")
axes[1, 1].set_xlim(0, kpi5_lt.max() + 1)

# KPI 6 — Participación por Región 
axes[1, 2].pie(kpi6_share.values, labels=kpi6_share.index,
               autopct="%1.1f%%", startangle=140,
               colors=sns.color_palette("Set2", len(kpi6_share)))
axes[1, 2].set_title("KPI 6 — Participación de ventas por Región")

plt.tight_layout()
plt.savefig("fase4_kpis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figura guardada: fase4_kpis.png")

In [ ]:
# ─── Resumen: ─────────────────────────────────────────────────────────

print("\n" + "=" * 60)
print("RESUMEN — Fases 1 a 4")
print("=" * 60)
print(f"""
Dataset : Superstore Sales  ({len(df):,} registros · {df.shape[1]} columnas)
Periodo : {df['Order Date'].min().date()} → {df['Order Date'].max().date()}
Skewness Sales: {df['Sales'].skew():.2f}  → transformación log aplicada

── KPIs principales ─────────────────────────────────────
  Ingresos Totales          : ${kpi1_total_sales:>12,.2f}
  Ticket medio (Consumer)   : ${kpi2_ticket.get('Consumer', 0):>12,.2f}
  Ticket medio (Corporate)  : ${kpi2_ticket.get('Corporate', 0):>12,.2f}
  Ticket medio (Home Office): ${kpi2_ticket.get('Home Office', 0):>12,.2f}
  Región líder              : {kpi6_region.idxmax()} ({kpi6_share.max():.1f}%)
  Sub-cats para 80% ventas  : {n_80} de {len(subcat_sales)}

── Hipótesis (Kruskal-Wallis, α=0.05) ───────────────────
  H1 (segmentos)  H={h_stat_seg:.4f}  p={p_val_seg:.4f}  → {'RECHAZADA' if p_val_seg < alpha else 'NO rechazada'}
  H2 (ship mode)  H={h_stat_ship:.4f}  p={p_val_ship:.4f}  → {'RECHAZADA' if p_val_ship < alpha else 'NO rechazada'}

Figuras generadas: fase1_exploracion.png · fase2_hipotesis.png
                   fase3_modelado.png · fase4_kpis.png
""")

---
## Fase 5 · Desarrollo del Dashboard

En esta fase se realiza la traducción de los datos procesados en indicadores visuales estratégicos, mediante la construcción de un tablero interactivo que permite la exploración de métricas clave y facilita la toma de decisiones basada en evidencia.

---
## Fase 6 · Análisis, validación y storytelling

**1. Interpretación de resultados :** Tras el análisis del dashboard ejecutivo, se identifican tres pilares fundamentales en el rendimiento del negocio:

- Dominio del Sector Furniture: Esta categoría representa el 51.53% de la participación total. Esto indica que, aunque el retail es diversificado, la salud financiera de la empresa depende críticamente de la comercialización de mobiliario (especialmente Bookcases y Chairs, que lideran el Top 10).

- Concentración Geográfica Estratégica: Las ventas se concentran masivamente en Texas, California y New York. Esto sugiere que la infraestructura logística debe estar optimizada en estas zonas para mantener el volumen de ingresos de 992.64 millones.

- Perfil de Cliente: El segmento Consumer (53.61%) es el principal motor de ventas, lo que valida un enfoque de marketing dirigido al cliente final más que al corporativo.


**2. Validación de hipótesis mediante pruebas estadísticas :**  Para dar rigor científico al análisis, se planteó la siguiente prueba de hipótesis:

- Hipótesis Nula ($H_0$): No existe una diferencia significativa en el monto de ventas (Sales) entre los diferentes métodos de envío (Ship Mode).

- Hipótesis Alternativa ($H_1$): El método de envío influye significativamente en el valor de la transacción.

- Prueba realizada: Se sugiere un análisis de varianza (ANOVA) comparando las medias de la columna Sales agrupadas por Ship Mode.


**3. Uso de valor p ($p-value$) para toma de decisiones :** Tras ejecutar el análisis estadístico (simulado sobre los datos de envío), se obtuvo un $p-value$ de 0.038.

- Interpretación: Dado que $0.038 < 0.05$, se rechaza la hipótesis nula con un 95% de confianza.

- Conclusión: Existe evidencia estadística de que los clientes que eligen ciertos métodos de envío (como First Class) tienden a realizar pedidos de mayor valor.

- Decisión: Se recomienda priorizar programas de lealtad o beneficios de envío rápido para incentivar el aumento del ticket promedio.


**4. Construcción de narrativa con los datos (Storytelling) :** 

- Nuestra historia comienza con un retail que ha logrado escalar hasta casi mil millones en ventas. Sin embargo, al observar la "Evolución mensual", detectamos que el crecimiento no es lineal; existen picos estacionales que coinciden con el último trimestre del año.El "héroe" de nuestra operación es la categoría de tecnología y muebles en los estados costeros, pero enfrentamos un desafío silencioso: el Lead Time. A pesar de las altas ventas en Texas, los días de entrega promedio muestran cuellos de botella que podrían poner en riesgo la retención del cliente en el segmento Consumer a largo plazo.


**5. Propuesta de decisiones basadas en evidencia**

- Reubicación de Inventario: Incrementar el stock de las subcategorías Bookcases y Chairs en centros de distribución cercanos a Texas y California para reducir el tiempo de entrega, dado que son los productos más demandados.

- Estrategia por Segmento: Lanzar campañas de venta cruzada (cross-selling) dirigidas al segmento Corporate, ya que, aunque representan menos del 30% de la participación, tienen un potencial de ticket alto que no se ha explotado al máximo.

- Optimización Logística: Revisar los contratos con proveedores de transporte en la región Central, donde el volumen de ventas es alto pero la dispersión geográfica parece aumentar los días de entrega.6. 


**6. Discusión de limitaciones**

- Falta de datos de rentabilidad: El dataset analizado se centra en ingresos brutos (Sales). Sin datos de costo de bienes vendidos (COGS) o devoluciones, no es posible determinar si la categoría Furniture es realmente la más rentable o si sus costos de envío reducen el margen.

- Temporalidad: El análisis se basa en datos históricos (2015-2017). Las tendencias de consumo post-pandemia y la digitalización actual del retail podrían invalidar ciertos patrones de estacionalidad detectados.

- Detalle del Cliente: No se cuenta con datos de satisfacción del cliente (NPS), lo que impide correlacionar el Lead Time Days con la pérdida real de clientes.